[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/05_cnn/05_cnn.ipynb)

# 05. Convolutional Neural Networks (CNN)

> 원본 강의: [Lec 11, 모두를 위한 머신러닝과 딥러닝](https://hunkim.github.io/ml/) — Convolution 레이어, Max Pooling, CNN으로 MNIST 99%+

## 이 장을 배우는 이유

04번 마지막에 이런 말을 했습니다. **"Flatten이 픽셀 사이의 위치 관계를 버린다."**
이번 장은 그 한 문장에서 출발합니다.

04번 모델은 28×28 이미지를 784개 숫자로 쭉 펴서 넣었습니다. 그러면 이런 일이 생깁니다.

- 왼쪽 위 픽셀과 그 바로 오른쪽 픽셀이 **이웃이라는 사실이 사라집니다**
- 숫자 3을 화면에서 **한 칸 오른쪽으로 옮기기만 해도** 784개 숫자가 통째로 달라집니다
- 사람은 "동그라미 두 개가 세로로 붙어 있으면 8"처럼 **모양**으로 읽는데, 모델은 그럴 수 없습니다

그래서 04번 모델은 96~97%에서 더 못 올라갑니다. 이번 장에서는 이미지를 **펴지 않고
2차원 그대로** 다루는 방법을 배웁니다.

이번 장에서 배우는 것

- 작은 창으로 이미지를 훑는 [합성곱](../../../glossary.md#convolution)(Convolution)
- 그 결과인 [특성 맵](../../../glossary.md#feature-map), 그리고 [스트라이드](../../../glossary.md#stride)·[패딩](../../../glossary.md#padding)
- 같은 필터를 온 이미지에 재사용하는 [가중치 공유](../../../glossary.md#weight-sharing)
- 크기를 줄여 계산을 아끼는 [풀링](../../../glossary.md#pooling)
- 이것들을 쌓아 만든 [CNN](../../../glossary.md#cnn)으로 MNIST 다시 풀기

**소요 시간**: 40~50분. 학습 셀은 CPU에서 3~6분 걸립니다(04번보다 조금 더 걸립니다).

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요**(`Shift + Enter`).
- **실행 결과는 저장되어 있지 않습니다.** 직접 실행해야 출력과 그래프가 나타납니다.
- 코드 셀 앞에는 **지금 무엇을 할 것인지**, 뒤에는 **결과를 어떻게 읽는지**를 적어두었습니다.
- **학습이 포함된 셀은 몇 분 걸립니다.** Colab이라면 `런타임 > 런타임 유형 변경`에서
  GPU를 켜면 훨씬 빨라집니다.
- 04번의 [PyTorch](../../../glossary.md#pytorch) 학습 루프를 알고 있다고 가정합니다.
- 낯선 용어는 [glossary.md](../../../glossary.md)에서 찾아보세요.


## 0. 준비


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q torch torchvision matplotlib koreanize-matplotlib


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader   # 데이터를 배치 단위로 잘라 공급해 주는 도구

try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("device:", device)


## 1. 문제 — Flatten이 정확히 무엇을 버리는가

말로 하면 와닿지 않으니 눈으로 봅시다. MNIST를 불러온 뒤, 이미지 한 장을 두 가지로 보여드립니다.

1. 원래 이미지
2. **784개 픽셀의 순서를 무작위로 섞은** 이미지

04번 모델 입장에서 **이 둘은 완전히 같은 난이도**입니다. 어차피 펴서 넣을 거라면
픽셀이 어느 자리에 있든 상관없기 때문입니다(순서만 일정하면 됩니다).
사람에게는 전혀 그렇지 않죠.


In [ ]:
transform = transforms.Compose([transforms.ToTensor()])   # 이미지를 0~1 사이 실수 텐서로 변환
train_ds = datasets.MNIST(root="../../../data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="../../../data", train=False, download=True, transform=transform)

img, label = train_ds[0]
flat = img.reshape(-1)                          # 28×28 -> 784 (04번의 Flatten이 하는 일)
perm = torch.randperm(flat.numel())             # 784개 자리를 무작위로 섞을 순서
shuffled = flat[perm].reshape(28, 28)           # 섞은 뒤 다시 28×28 모양으로 되돌려 보기

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img.squeeze(), cmap="gray")
axes[0].set_title(f"원래 이미지 (정답 {label})")
axes[1].imshow(shuffled, cmap="gray")
axes[1].set_title("픽셀 순서를 섞은 이미지")
for ax in axes:
    ax.axis("off")
plt.show()

print(f"두 이미지의 픽셀값 합계: {flat.sum():.2f} vs {shuffled.sum():.2f}   <- 완전히 같다")


**결과 읽는 법** — 오른쪽은 사람 눈에 그냥 잡음입니다. 그런데 **들어 있는 숫자 784개는 똑같고,
순서만 다릅니다.** 픽셀값의 합계가 같은 것이 그 증거입니다.

04번 모델은 오른쪽 이미지로 학습시켜도 정확도가 거의 똑같이 나옵니다.
**"이 픽셀 옆에 저 픽셀이 있다"는 정보를 애초에 쓰지 않기 때문**입니다.

우리가 원하는 것은 반대입니다. **이웃한 픽셀들을 묶어서 보는 모델.**


## 2. 아이디어 — 작은 창으로 훑어보기

이미지 전체를 한 번에 보지 말고, **3×3짜리 작은 창을 이미지 위로 한 칸씩 옮겨가며**
그 안의 9개 픽셀만 보면 어떨까요? 그러면 "이웃한 픽셀끼리"를 자연스럽게 함께 보게 됩니다.

이것이 [합성곱](../../../glossary.md#convolution)(Convolution)입니다. 창 안에서 하는 계산은 단순합니다.
**9개 픽셀에 각각 정해진 수를 곱해서 더합니다.** 그 "정해진 수 9개"를 **필터(filter)** 또는
**커널(kernel)** 이라고 부릅니다.

필터를 어떻게 정하느냐에 따라 뽑아내는 특징이 달라집니다. 예를 들어 이런 필터는

```
-1  0  +1
-1  0  +1
-1  0  +1
```

왼쪽은 빼고 오른쪽은 더합니다. 왼쪽과 오른쪽 밝기가 비슷한 곳(=밋밋한 부분)에서는 0이 되고,
**왼쪽이 어둡고 오른쪽이 밝은 곳(=세로 경계선)에서만 큰 값**이 나옵니다.
즉 이 필터는 **세로 경계선 탐지기**입니다.

직접 확인해봅시다. 왼쪽 절반이 검고 오른쪽 절반이 흰 8×8 이미지를 만들어 통과시킵니다.


In [ ]:
# Conv2d는 항상 배치·채널 축까지 포함한 4차원을 기대하므로, 8×8 이미지 한 장도 이 모양으로 감싸야 한다
img8 = torch.zeros(1, 1, 8, 8)
img8[:, :, :, 4:] = 1.0        # 오른쪽 절반만 밝게

vertical_edge_filter = torch.tensor([[[[-1., 0., 1.],
                                       [-1., 0., 1.],
                                       [-1., 0., 1.]]]])

# Conv2d(입력 채널, 출력 채널, 커널 크기): 3×3 필터를 이미지 위로 훑는 층
conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
with torch.no_grad():
    conv.weight.copy_(vertical_edge_filter)   # 보통은 학습으로 정해지지만, 여기서는 직접 넣어본다

feature_map = conv(img8)

print(f"입력 크기: {tuple(img8.shape[-2:])}  ->  출력 크기: {tuple(feature_map.shape[-2:])}")
print("\n출력값(특성 맵):")
print(feature_map[0, 0].detach().numpy().round(1))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img8[0, 0], cmap="gray")
axes[0].set_title("입력 이미지 (8×8)")
axes[1].imshow(feature_map[0, 0].detach(), cmap="gray")
axes[1].set_title("합성곱 결과 (특성 맵)\n세로 경계선만 남는다")
plt.show()

**결과 읽는 법** — 출력 숫자를 보면 **대부분 0이고, 경계가 있던 열에서만 3이 나옵니다.**
필터가 "여기 세로 경계선이 있다"고 표시한 것입니다. 이 결과물을 [특성 맵](../../../glossary.md#feature-map)
(feature map)이라고 부릅니다.

**크기가 8×8에서 6×6으로 줄었습니다.** 3×3 창을 이미지 안에만 놓으려면 가장자리에서는
창이 밖으로 나가버리기 때문입니다. 이걸 막으려면 이미지 테두리에 0을 한 겹 둘러주면 되고,
그것을 [패딩](../../../glossary.md#padding)(padding)이라고 합니다. `padding=1`을 주면 크기가 유지됩니다.

또 하나. 창을 **한 칸씩** 옮길지 **두 칸씩** 옮길지도 정할 수 있는데, 그 보폭을
[스트라이드](../../../glossary.md#stride)(stride)라고 합니다. 두 칸씩 가면 출력 크기가 절반이 됩니다.

**여기서 CNN의 진짜 강점이 나옵니다.** 방금 그 필터는 숫자가 **9개뿐**인데, 이미지의
모든 위치에 **똑같이** 적용됐습니다. 왼쪽 위에서 경계선을 찾던 그 필터가 오른쪽 아래에서도
그대로 경계선을 찾습니다. 이것을 [가중치 공유](../../../glossary.md#weight-sharing)라고 합니다.

- 04번 MLP는 왼쪽 위 픽셀용 가중치와 오른쪽 아래 픽셀용 가중치가 **따로**였습니다
- CNN은 **필터 하나를 온 이미지에서 재사용**합니다

그래서 **파라미터가 훨씬 적고**, **숫자가 화면에서 조금 움직여도 잘 견딥니다.**


## 3. 크기 줄이기 — 풀링

합성곱을 여러 번 하면 특성 맵이 계속 쌓여 계산량이 커집니다. 그런데 생각해보면,
"이 근처 어딘가에 세로 경계선이 있다"만 알면 되지 **정확히 몇 번째 픽셀인지까지는 필요 없습니다.**

그래서 2×2 구역마다 **가장 큰 값 하나만 남기고 버립니다.** 이것이
[맥스 풀링](../../../glossary.md#pooling)(Max Pooling)이고, 크기가 정확히 절반이 됩니다.

- 28×28 → `MaxPool2d(2)` → 14×14 → 다시 → 7×7

부수 효과가 하나 더 있습니다. 숫자가 한두 픽셀 움직여도 **구역 안의 최댓값은 잘 안 바뀝니다.**
그래서 위치 변화에 조금 더 둔감해집니다.


## 4. 조립 — CNN 만들기

이제 부품이 다 모였습니다. 전형적인 CNN은 이 순서로 쌓습니다.

```
[Conv -> ReLU -> Pool]  ...여러 번 반복...  -> Flatten -> Linear -> 출력
 └───── 특징을 뽑는 부분 ─────┘              └── 뽑은 특징으로 판단하는 부분 ──┘
```

**앞부분이 "무엇이 보이는지"를 찾고, 뒷부분이 "그래서 무슨 숫자인지"를 답합니다.**
Flatten은 여전히 쓰이지만, 이제는 **원본 픽셀이 아니라 이미 위치 관계를 반영해 뽑아낸 특징**을
펴는 것이라 정보 손실이 훨씬 적습니다.

아래 코드에서 **주석의 크기 변화(28×28 → 14×14 → 7×7)를 눈으로 따라가 보세요.**


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 28x28 -> 28x28 (padding=1이라 크기 유지), 필터 16종류
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28x28 -> 14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 14x14 -> 14x14, 필터 32종류
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # 32채널 × 7 × 7 = 1568개를 한 줄로
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)        # 특징 뽑기
        return self.classifier(x)   # 뽑은 특징으로 판단


model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

feat_params = sum(p.numel() for p in model.features.parameters())      # numel(): 텐서 안의 원소 개수
clf_params = sum(p.numel() for p in model.classifier.parameters())
print(f"특징 추출부(Conv 2개) 파라미터: {feat_params:>9,}")
print(f"판단부(Linear 2개) 파라미터  : {clf_params:>9,}")
print(f"합계                          : {feat_params + clf_params:>9,}")


**결과 읽는 법 — 이 숫자를 꼭 보세요.**

- **Conv 층 2개가 쓰는 파라미터는 5천 개도 안 됩니다.** 이미지의 특징을 찾아내는 핵심 역할을
  이렇게 적은 숫자로 해냅니다. 가중치 공유 덕분입니다.
- 반면 **뒤쪽 Linear 층 하나가 20만 개 넘게** 씁니다. 1568개를 128개로 잇는 데 그만큼 필요합니다.
- 04번 MLP의 파라미터는 약 23만 개였습니다. 전체 규모는 비슷한데,
  **CNN은 그중 대부분을 판단부에 쓰고 특징 추출은 거의 공짜로 합니다.**

이제 학습입니다. 학습 루프는 **04번과 글자 하나 다르지 않습니다.** 모델만 바뀌었습니다.
PyTorch에서 모델 종류가 달라져도 학습 코드는 그대로라는 점을 확인해두세요.


In [ ]:
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return correct / total


EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)

    train_loss = running_loss / len(train_ds)
    test_acc = evaluate(model, test_loader)
    print(f"epoch {epoch + 1}/{EPOCHS}  train_loss={train_loss:.4f}  test_acc={test_acc:.4f}")


**결과 읽는 법**

- 같은 3바퀴인데 **04번 MLP(96~97%)보다 높은 98~99%대**가 나옵니다.
  1바퀴만에 이미 04번의 3바퀴 성적을 넘는 경우가 많습니다.
- 틀리는 개수로 보면 차이가 더 실감납니다. 1만 장 중 300장 틀리던 것이 100장대로 줄어듭니다.

**이럴 때는 이걸 의심하세요.**

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| MLP보다 느린데 정확도가 비슷하다 | 1 epoch만 돌린 상태 | 3바퀴까지 돌려보기 |
| 학습이 너무 느리다 | CPU로 도는 중 | Colab에서 GPU 켜기 |
| 크기 관련 에러가 난다 | `Linear`의 입력 크기 불일치 | `32 * 7 * 7` 계산을 층 구조와 대조 |

**왜 더 잘할까요?** 1절에서 본 그 이유입니다. CNN은 픽셀을 펴지 않고 이웃끼리 묶어 보기 때문에,
"획이 어떻게 생겼는지"를 볼 수 있습니다. 그리고 그 필터를 이미지 전체에서 재사용하므로
숫자가 조금 치우쳐 쓰여 있어도 같은 특징을 찾아냅니다.


## 정리

이번 장에서 한 일

1. Flatten이 버리는 것이 무엇인지 **픽셀을 섞어 눈으로** 확인했습니다
2. 3×3 창으로 이웃 픽셀을 함께 보는 **합성곱**을 배웠습니다
3. 필터 하나를 온 이미지에서 재사용하는 **가중치 공유**가 왜 강력한지 파라미터 수로 봤습니다
4. **풀링**으로 크기를 줄이고 위치 변화에 둔감해졌습니다
5. `[Conv → ReLU → Pool]`을 쌓아 MNIST 정확도를 **98~99%대**로 올렸습니다

**스스로 확인해보기**

- [ ] 픽셀을 섞은 이미지가 MLP에게는 왜 똑같은 난이도인지 설명할 수 있다
- [ ] 필터가 무엇을 하는 숫자 9개인지 말할 수 있다
- [ ] `padding=1`을 왜 주는지 안다
- [ ] Conv 층의 파라미터가 왜 그렇게 적은지 안다
- [ ] 풀링이 크기 말고 무엇을 더 얻어주는지 안다

## 연습 문제

1. 같은 3 epoch 기준으로 04번 MLP와 정확도를 나란히 비교해보세요. 몇 장을 더 맞혔나요?
2. `Conv2d`의 채널 수(16, 32)나 층 수를 늘려보고 정확도와 학습 시간이 어떻게 달라지는지 보세요.
3. 2절의 필터를 가로 경계선 탐지기로 바꿔보세요(행과 열을 뒤집으면 됩니다).
   그리고 위아래로 나뉜 이미지에 적용하면 어떻게 되나요?

**해설/정답**: [05_cnn_solutions.ipynb](05_cnn_solutions.ipynb)

다음 노트북([06_rnn.ipynb](../06_rnn/06_rnn.ipynb))에서는 이미지가 아니라
**순서가 있는 데이터**(문장, 시계열)를 다룹니다. 이번에 잃지 않으려 애쓴 것이 "위치"였다면,
다음에 지켜야 할 것은 "순서"입니다.
